In [1]:
import polars as pl
from tqdm import tqdm
import re
import os
import logging

In [2]:
# icd and polars testing

In [3]:
# new function
def icd_to_stage(icd):
    """
        'N18.1%': 1,
        'N18.2%': 2, 
        'N18.3%': 3, 
        'N18.4%': 4, 
        'N18.5%': 5, 
        'N18.6%': 'ESRD', 
        'N18.9%': 'CKD'
    """
    if icd is None:
        return (None, 0)
    if icd in 'N18.1%':
        return ("1", 1)
    if icd in 'N18.2%':
        return ("2", 2)
    if icd in 'N18.3%':
        return ("3", 3)
    if icd in 'N18.4%':
        return ("4", 4)
    return ("5", 5)

In [4]:
# Setup logging
output_path = "./../../../commonfilesharePHI/ldiao/ckd_project/"
output_dir_m = output_path + "ckd_tab_m_full" # << change
output_fname = "ckd_processed_tab.csv"
log_fname = "tab_gen_log.log"
event_file = "/opt/data/commonfilesharePHI/jnchiang/projects/OptumCKD/CKD-Pull_v2.rpt"
event_file2 = "./../../../commonfilesharePHI/slee/ckd-optum/patients_subset_10.csv" # 10, 100, all


In [5]:
# df = pl.read_csv(event_file, separator='$', infer_schema_length=10000).unique()
df = pl.read_csv(event_file2, infer_schema_length=10000).unique()
# logger.info(f"Initial DataFrame shape: {df.shape}")

In [ ]:
df.head()

In [7]:
# df = df.with_columns(
#     pl.col('EventTimeStamp').str.strptime(pl.Datetime, '%Y-%m-%d %H:%M:%S.%f', strict=False),
# )

In [ ]:
df.head()

In [ ]:
df = (
    df.with_columns(
        pl.col('EventTimeStamp').str.strptime(pl.Datetime, '%Y-%m-%d %H:%M:%S.%f', strict=False),
    )
    .with_columns(
        pl.col('EventTimeStamp').dt.date().alias('EventDate'),
        pl.col('DataCategory').fill_null('None'),
        pl.col('DataNumeric').cast(pl.Float64, strict=False),
    )
)

In [ ]:
df.head()

In [ ]:
df.shape

In [12]:
# -----------------------------
# Base: full patient-day index
# -----------------------------
df = df.with_columns(
    pl.col('DataCategory').str.to_uppercase().str.contains("GFR|GFREST").alias("is_gfr")
)

all_days = (
    df.select(['PatientID', 'EventDate'])
    .unique()
    .sort(['PatientID', 'EventDate'])
    .drop_nulls()
)

In [ ]:
df.head()

In [14]:

# -----------------------------
# Extract and forward-fill GFR
# -----------------------------
gfr_df = df.filter(pl.col('is_gfr') & pl.col('DataNumeric').is_not_null())

gfr_daywise = (
    gfr_df.group_by(['PatientID', 'EventDate'])
    .agg(pl.col('DataNumeric').first().alias('GFR_combined'))
)

base_df = all_days.join(gfr_daywise, on=['PatientID', 'EventDate'], how='left')

base_df = base_df.sort(['PatientID', 'EventDate'])
base_df = base_df.with_columns(
    pl.col('GFR_combined').forward_fill().over('PatientID')
)

In [ ]:
base_df.head()

In [16]:
def gfr_to_stage(gfr):
    if gfr is None:
        return (None, 0)
    if gfr >= 90:
        return ("1", 1)
    if gfr >= 60:
        return ("2", 2)
    if gfr >= 45:
        return ("3a", 3.1)
    if gfr >= 30:
        return ("3b", 3.2)
    if gfr >= 15:
        return ("4", 4)
    return ("5", 5)


In [17]:
# apply gfr_to stage to column
# split output column into two columns one for stage, one for rank
# then enforce monotonic ckd staging

In [ ]:
import pandas as pd
def gfr_to_stage(gfr):
    if pd.isna(gfr) is None:
        return None, 0
    if gfr is None:
        return None, 0
    if gfr >= 90:
        return "1", 1
    if gfr >= 60:
        return "2", 2
    if gfr >= 45:
        return "3a", 3.1
    if gfr >= 30:
        return "3b", 3.2
    if gfr >= 15:
        return "4", 4
    return "5", 5


def enforce_monotonic_ckd(group):
    # Sort the group by date
    group = group.sort("EventDate")
    
    # Initialize state variables for the loop
    max_rank = -1  # Initialize with a value lower than any possible rank
    prev_stage = None
    new_stages = []
    
    for row in group.iter_rows(named=True):
        gfr = row["GFR_combined"]
        current_stage, current_rank = gfr_to_stage(gfr)
        
        # This is where the core logic is corrected
        # If the current rank is lower than the max rank seen so far
        if current_rank < max_rank:
            # Use the previous valid stage
            stage_to_append = prev_stage
        else:
            # Otherwise, the rank has increased or stayed the same
            stage_to_append = current_stage
            max_rank = current_rank
            
        # Update the previous stage for the next iteration
        prev_stage = stage_to_append
        new_stages.append(stage_to_append)
        
    return group.with_columns(pl.Series(new_stages).alias("CKD_stage"))

# Apply the custom function to each group
result_df = base_df.group_by("PatientID").map_groups(enforce_monotonic_ckd)



In [ ]:
print(result_df)